# Deletar as tabelas casos existam

Serve para limpar o ambiente antes de criar as tabelas

In [0]:
%sql
DROP TABLE IF EXISTS mvp_engenharia_dados.gold.dim_localidade;
DROP TABLE IF EXISTS mvp_engenharia_dados.gold.dim_causa;
DROP TABLE IF EXISTS mvp_engenharia_dados.gold.fato_obitos;


# Criação das tabelas Gold

A camada Gold tem como objetivo transformar os dados tratados na camada Silver em uma estrutura otimizada para responder às perguntas de negócio do MVP. Diferente da Silver, que organiza e garante a qualidade dos dados, a Gold é modelada pensando diretamente no consumo analítico.

Para isso, foi adotado um **Esquema Estrela (Star Schema)**, composto por uma tabela fato central e tabelas dimensão que descrevem o contexto de cada óbito:
- **`fato_obitos`**: tabela fato, contendo um registro por óbito e as métricas/atributos necessários para as análises;
- **`dim_localidade`**: tabela dimensão com os municípios e estados, permitindo identificar e comparar diferentes localidades (Salvador, Bahia);
- **`dim_causa`**: tabela dimensão com os códigos CID-10 e seus respectivos capítulos, permitindo identificar e agrupar as causas de óbito.

Os atributos `SEXO`, `FAIXA_ETARIA` e `ANO_OBITO` foram mantidos diretamente na tabela fato, e não como dimensões separadas. Essa decisão foi tomada porque são atributos de
baixa cardinalidade (poucos valores possíveis) e não possuem informações adicionais que justifiquem uma tabela própria.

Diferente de uma abordagem com tabelas pré-agregadas para cada pergunta, esse desenho permite que qualquer combinação de filtros e agrupamentos (por ano, sexo, faixa etária,
causa ou localidade) seja feita diretamente sobre a `fato_obitos`, sem necessidade de recriar tabelas para cada nova análise. Como o volume de dados do projeto é pequeno
(óbitos da Bahia entre 2020 e 2025), não há ganho de performance relevante em pré-agregar os dados nesta etapa (a agregação final será feita nas consultas da etapa de Análise).

## Perguntas que a estrutura permite responder

- Quantos óbitos ocorreram em Salvador entre 2020 e 2025, e quais as principais causas?
- Como os óbitos se distribuem por sexo e faixa etária?
- Como a mortalidade de Salvador se compara à da Bahia?
- Como os óbitos e suas principais causas evoluíram ao longo dos anos?

## Processo de construção

As dimensões `dim_localidade` e `dim_causa` são construídas a partir das tabelas já tratadas na camada Silver (`municipio`+`estado` e `cid`+`capitulo_cid`, respectivamente),
unificando as informações relacionadas em uma única tabela por tema.

A tabela `fato_obitos` é construída a partir da tabela `silver.sim`, relacionando cada óbito à sua localidade de residência (via `CODMUNRES` → `CODIGO_MUNICIPIO_SAUDE`) e à
sua causa básica (via `CAUSABAS_GERAL` → `CODIGO_CID`). Após o relacionamento, é verificado se existem óbitos sem correspondência nas dimensões, o que indicaria problemas de integridade entre as bases.

## Tratamento de registros sem correspondência nas dimensões

Durante a construção da tabela fato, foram identificados óbitos cujo código de município de residência ou código de causa básica não possuem correspondência nas tabelas dimensão. Isso ocorre por dois motivos:

- **Localidade**: parte dos códigos de município não corresponde a nenhum município válido na tabela do IBGE (possivelmente códigos de preenchimento inválido/ignorado ou municípios extintos);
- **Causa**: a base de CID utilizada no projeto tem origem em uma planilha de concessões de benefício por auxílio-doença do Ministério da Previdência Social, e não em uma tabela de referência oficial e completa do CID-10. Por esse motivo, ela não contém causas que raramente geram esse tipo de benefício, como causas externas (acidentes, agressões, óbitos por causas violentas). Essa limitação é assumida como uma restrição conhecida da base utilizada neste MVP.

Para que esses registros não sejam silenciosamente descartados nas análises, foi adicionada uma linha "Não identificado" em cada dimensão, e os registros sem correspondência foram associados a ela.

In [0]:
from pyspark.sql import functions as F

# ============================================================
# Construção da dimensão de localidade
# ============================================================

municipio = spark.table("mvp_engenharia_dados.silver.municipio")
estado = spark.table("mvp_engenharia_dados.silver.estado")

dim_localidade = (
    municipio
    .join(estado, on="ID_ESTADO", how="left")
    .select(
        "ID_MUNICIPIO",
        "CODIGO_MUNICIPIO_SAUDE",
        "NOME_MUNICIPIO",
        "CODIGO_UF",
        "NOME_UF"
    )
    .withColumn(
        "FLAG_SALVADOR",
        F.when(
            (F.col("NOME_MUNICIPIO") == "Salvador") &
            (F.col("NOME_UF") == "Bahia"),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
)

display(dim_localidade)

# salvar a tabela na camada gold
dim_localidade.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_engenharia_dados.gold.dim_localidade")

In [0]:
# ============================================================
# Construção da dimensão de causa (CID)
# ============================================================

cid = spark.table("mvp_engenharia_dados.silver.cid")
capitulo = spark.table("mvp_engenharia_dados.silver.capitulo_cid")

dim_causa = (
    cid
    .join(capitulo, on="ID_CAPITULO", how="left")
    .select(
        "ID_CID",
        "CODIGO_CID",
        "NOME_CID",
        "NOME_CAPITULO"
    )
)

display(dim_causa)

# salvar a tabela na camada gold
dim_causa.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_engenharia_dados.gold.dim_causa")

In [0]:
# ============================================================
# Construção da tabela fato de óbitos
# ============================================================

sim = spark.table("mvp_engenharia_dados.silver.sim")

fato_obitos = (
    sim
    .join(
        dim_localidade.select(
            F.col("ID_MUNICIPIO").alias("ID_MUNICIPIO_RESIDENCIA"),
            F.col("CODIGO_MUNICIPIO_SAUDE")
        ),
        sim.CODMUNRES == F.col("CODIGO_MUNICIPIO_SAUDE"),
        how="left"
    )
    .join(
        dim_causa.select(
            F.col("ID_CID").alias("ID_CAUSA"),
            F.col("CODIGO_CID")
        ),
        sim.CAUSABAS_GERAL == F.col("CODIGO_CID"),
        how="left"
    )
    .select(
        "IDOBITO",
        "ANO_OBITO",
        "SEXO",
        "FAIXA_ETARIA",
        "ID_MUNICIPIO_RESIDENCIA",
        "ID_CAUSA"
    )
    .withColumn("QTD_OBITO", F.lit(1))
)

display(fato_obitos)

sem_localidade = fato_obitos.filter(F.col("ID_MUNICIPIO_RESIDENCIA").isNull()).count()
sem_causa = fato_obitos.filter(F.col("ID_CAUSA").isNull()).count()

print(f"Óbitos sem município de residência associado: {sem_localidade}")
print(f"Óbitos sem causa associada: {sem_causa}")


# salvar a tabela na camada gold
fato_obitos.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_engenharia_dados.gold.fato_obitos")

In [0]:
# ============================================================
# Adicionar registro "Não identificado" nas dimensões
# ============================================================

from pyspark.sql import Row
from pyspark.sql import functions as F

dim_localidade = spark.table("mvp_engenharia_dados.gold.dim_localidade")
dim_causa = spark.table("mvp_engenharia_dados.gold.dim_causa")

localidade_ni = spark.createDataFrame([
    Row(ID_MUNICIPIO=-1, CODIGO_MUNICIPIO_SAUDE="000000",
        NOME_MUNICIPIO="Não identificado", CODIGO_UF="00",
        NOME_UF="Não identificado", FLAG_SALVADOR=False)
])

# Ajustar os tipos para bater exatamente com o schema da tabela original
localidade_ni = localidade_ni.select(
    F.col("ID_MUNICIPIO").cast(dim_localidade.schema["ID_MUNICIPIO"].dataType),
    F.col("CODIGO_MUNICIPIO_SAUDE"),
    F.col("NOME_MUNICIPIO"),
    F.col("CODIGO_UF"),
    F.col("NOME_UF"),
    F.col("FLAG_SALVADOR")
)

causa_ni = spark.createDataFrame([
    Row(ID_CID=-1, CODIGO_CID="000",
        NOME_CID="Não identificado", NOME_CAPITULO="Não identificado")
])

causa_ni = causa_ni.select(
    F.col("ID_CID").cast(dim_causa.schema["ID_CID"].dataType),
    F.col("CODIGO_CID"),
    F.col("NOME_CID"),
    F.col("NOME_CAPITULO")
)

dim_localidade = dim_localidade.unionByName(localidade_ni)
dim_causa = dim_causa.unionByName(causa_ni)

dim_localidade.write.format("delta").mode("overwrite") \
    .saveAsTable("mvp_engenharia_dados.gold.dim_localidade")
dim_causa.write.format("delta").mode("overwrite") \
    .saveAsTable("mvp_engenharia_dados.gold.dim_causa")

In [0]:
# ============================================================
# Reconstruir a fato apontando nulos para "Não identificado"
# ============================================================

from pyspark.sql import functions as F

fato_obitos = spark.table("mvp_engenharia_dados.gold.fato_obitos")

fato_obitos = (
    fato_obitos
    .withColumn(
        "ID_MUNICIPIO_RESIDENCIA",
        F.coalesce(F.col("ID_MUNICIPIO_RESIDENCIA"), F.lit(-1))
    )
    .withColumn(
        "ID_CAUSA",
        F.coalesce(F.col("ID_CAUSA"), F.lit(-1))
    )
)

fato_obitos.write.format("delta").mode("overwrite") \
    .saveAsTable("mvp_engenharia_dados.gold.fato_obitos")

In [0]:
# verificar tabela sim da camada gold
display(
    spark.table("mvp_engenharia_dados.gold.fato_obitos")
)

display(
    spark.table("mvp_engenharia_dados.gold.dim_causa")
)

display(
    spark.table("mvp_engenharia_dados.gold.dim_localidade")
)